# 02 — Generative Model Comparison

## 📚 Learning Objectives

By completing this notebook, you will:
- Compare the three main deep generative families — **GANs**, **VAEs**, and **flow-based models** — on training objective, sample quality, diversity, speed, and likelihood
- Practice **selecting a model family** for a given use case with an explicit decision procedure

## 🔗 Where this fits

**Builds on:** Course 08 (AIAT 122) — Unit 4, lesson 01 "GANs and Autoencoders / VAEs", where these families were first met, and Course 10 — Unit 1, lesson 01: generative is now the chosen side, so the question is which family.

**Used later in:** Course 10 — Units 1-3, which implement each family in turn, and Unit 5, lesson 04, which compares their production descendants.

---

## Introduction

All generative models answer the same question — *"what does the data distribution look like, and can I sample from it?"* — but the three big families answer it very differently. This notebook is a **conceptual comparison and selection guide**; the hands-on implementations follow in example 04 (VAE) and example 05 (GAN).

---

## 🌍 Why this lesson exists — the family choice that decided the 2022 image-generation market

This selection decision was made for real, publicly, in 2022. Stability AI shipped **Stable Diffusion** as *latent diffusion* (Rombach et al., 2022, arXiv 2112.10752), not as a GAN, while NVIDIA's StyleGAN line stayed confined to narrow aligned domains such as faces. By 2026 essentially every production image and video generator descends from that diffusion lineage — transformer diffusion backbones (Peebles & Xie, 2022, arXiv 2212.09748) and flow matching (Lipman et al., 2022, arXiv 2210.02747) — not from the GAN lineage. Read the table below with that in mind: it is an accurate map of 2014–2021, and the family that won production is the one it does not list. Diffusion arrives in Unit 3, example 04.

**What goes wrong without this:** teams pick a family by folklore ("GANs are sharp") instead of by requirement. Three months later the training loop still will not converge, and someone finally states the real requirement — a likelihood, or a text prompt, or a 200 ms latency budget — that the chosen architecture cannot supply at any amount of tuning.


## The Three Families

**GAN (Generative Adversarial Network)** — a Generator is trained to fool a Discriminator (example 01 previewed this; example 05 builds it). No explicit probability model: you cannot ask a GAN "how likely is this image?". Samples tend to be **sharp** but training is unstable and can *mode-collapse* (produce only a few types of output).

**VAE (Variational Autoencoder)** — an encoder maps data to a latent Gaussian distribution, a decoder maps latent vectors back to data; trained by maximizing a lower bound (ELBO) on log-likelihood (example 04 builds one). Training is **stable** and the latent space is smooth and useful, but samples tend to be **blurrier** because of the pixel-wise reconstruction loss.

**Flow-based models** (RealNVP, Glow) — a chain of *invertible* transformations turns a simple Gaussian into the data distribution. Because every step is invertible, the model computes **exact log-likelihood** — the only family that can. The price: architectural restrictions (invertibility) and typically larger/slower models. Flows are not implemented in this course; see the references.


In [1]:
# WHAT/WHY: put the three families side by side in one comparison table.
# This is reference knowledge (from the papers cited below), organized so you
# can scan a row and see how the families differ on each property.
import pandas as pd

comparison = pd.DataFrame({
    "GAN":  ["adversarial game (fool a discriminator)", "implicit — none",
             "sharp", "can mode-collapse (low)", "fast (one forward pass)",
             "unstable, needs tricks (example 08)"],
    "VAE":  ["maximize ELBO (reconstruction + KL)", "lower bound (ELBO)",
             "blurrier", "good coverage", "fast (one decoder pass)",
             "stable"],
    "Flow": ["maximize exact log-likelihood", "exact",
             "moderate", "good coverage", "fast to sample, costly to train",
             "stable but architecture-restricted"],
}, index=["Training objective", "Likelihood available?", "Typical sample quality",
          "Sample diversity", "Sampling speed", "Training stability"])

# Show the full table without truncation
with pd.option_context("display.max_colwidth", 60, "display.width", 140):
    print(comparison.to_string())


                                                            GAN                                  VAE                                Flow
Training objective      adversarial game (fool a discriminator)  maximize ELBO (reconstruction + KL)       maximize exact log-likelihood
Likelihood available?                           implicit — none                   lower bound (ELBO)                               exact
Typical sample quality                                    sharp                             blurrier                            moderate
Sample diversity                        can mode-collapse (low)                        good coverage                       good coverage
Sampling speed                          fast (one forward pass)              fast (one decoder pass)     fast to sample, costly to train
Training stability          unstable, needs tricks (example 08)                               stable  stable but architecture-restricted


In [2]:
# WHAT/WHY: turn the table into an explicit decision procedure — a small
# rule-based function that recommends a model family from a use case's
# requirements, then run it on three realistic scenarios.

def recommend_family(needs_exact_likelihood, needs_sharpest_samples, needs_easy_training):
    """Pick a generative family from three yes/no requirements (simplified guide)."""
    # Exact likelihood is the deciding requirement: only flows provide it.
    if needs_exact_likelihood:
        return "Flow", "only flow models give exact log-likelihood"
    # Next priority: photorealistic sharpness points to GANs.
    if needs_sharpest_samples and not needs_easy_training:
        return "GAN", "adversarial training yields the sharpest samples (at the cost of stability)"
    # Otherwise the stable, general-purpose default is the VAE.
    return "VAE", "stable training, smooth latent space, good-enough samples"

scenarios = [
    ("Density estimation / anomaly scoring for fraud detection",
     dict(needs_exact_likelihood=True,  needs_sharpest_samples=False, needs_easy_training=True)),
    ("Photorealistic face generation for a game studio",
     dict(needs_exact_likelihood=False, needs_sharpest_samples=True,  needs_easy_training=False)),
    ("Latent-space data exploration + controllable generation, small team",
     dict(needs_exact_likelihood=False, needs_sharpest_samples=False, needs_easy_training=True)),
]

# Apply the rules to each scenario and print the computed recommendation
for name, reqs in scenarios:
    family, reason = recommend_family(**reqs)
    print(f"Use case: {name}")
    print(f"  → Recommended family: {family}  (why: {reason})\n")


Use case: Density estimation / anomaly scoring for fraud detection
  → Recommended family: Flow  (why: only flow models give exact log-likelihood)

Use case: Photorealistic face generation for a game studio
  → Recommended family: GAN  (why: adversarial training yields the sharpest samples (at the cost of stability))

Use case: Latent-space data exploration + controllable generation, small team
  → Recommended family: VAE  (why: stable training, smooth latent space, good-enough samples)



## 💬 Discuss

1. Re-run the three scenarios in your head with **diffusion** added as a fourth option. Which recommendations change, and which line of `recommend_family` would you have to rewrite first? What requirement would you need to add to the function signature?
2. The rule checks `needs_exact_likelihood` first and returns Flow immediately. Is that the right priority for fraud scoring, where you also have to score ten thousand transactions a second? Reorder the checks and justify the new order.
3. The table's cell "GAN — sample diversity: can mode-collapse (low)" is a claim from the literature, not a measurement in this notebook. Which printed number in example 08 or example 09 would you point at to support it — and which one arguably contradicts it?


## Where You Will Build Them

- **GAN** — previewed in example 01; built step by step in **example 05**; stabilized in **example 08**; compared head-to-head with a VAE in **example 09**.
- **VAE** — built in **example 04**; latent-space exploration in **example 12**; applied in depth in Unit 3.
- **Flow models** — concept-level only in this course; start with the Glow paper below if you want to go further.


## ⚠️ Where this breaks

**Family-level tables are decision aids, not evidence.** Lucic et al. (2018, arXiv 1711.10337) trained the major GAN variants under a matched computational budget and reported no evidence that any of them consistently outperformed the original non-saturating GAN. Differences that look architectural in a table often turn out to be tuning and compute in a controlled comparison.

**The rows that usually decide are missing.** There is no row for training compute, data volume, latency, or licence. A team with 5,000 images and one GPU is not really choosing between families at all — the correct answer is to fine-tune an open checkpoint, which this table cannot recommend because it does not model "someone else already trained it".

**The table stops in 2021.** A 2026 selection guide that offers only GAN / VAE / flow will send you to the wrong architecture for image, audio and video work. Use it to understand *what the three objectives buy you*; do not use it as a procurement checklist.


## 📚 References & Further Reading

**Foundational Papers:**
- Goodfellow et al. (2014) — [Generative Adversarial Nets](https://arxiv.org/abs/1406.2661)
- Kingma & Welling (2014) — [Auto-Encoding Variational Bayes (VAE)](https://arxiv.org/abs/1312.6114)
- Dinh et al. (2017) — [RealNVP: Density Estimation Using Real NVP](https://arxiv.org/abs/1605.08803)
- Kingma & Dhariwal (2018) — [Glow: Generative Flow with Invertible 1×1 Convolutions](https://arxiv.org/abs/1807.03039)

**Survey:**
- Bond-Taylor et al. (2021) — [Deep Generative Modelling: A Comparative Review](https://arxiv.org/abs/2103.04922)


## 📝 Summary

In this notebook you compared the **three deep generative families**: GANs (adversarial, sharp, unstable, no likelihood), VAEs (ELBO, stable, smooth latents, blurrier), and flows (exact likelihood, restricted architectures). You also applied an explicit decision procedure to three realistic use cases. The hands-on builds come next: VAE in example 04, GAN in example 05.
